<a href="https://colab.research.google.com/github/AAwaisYaseen/computer-vision-logistics/blob/main/COMP6011_benchmarking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# COMP6011 - Task 1 Benchmarking Experiments
# Semantic Segmentation on SydneyScapes Dataset
# Models: DeepLabv3+ and PIDNet
#
# References:
# - DeepLabv3+: VainF (2021). DeepLabV3Plus-Pytorch.
#   https://github.com/VainF/DeepLabV3Plus-Pytorch
# - PIDNet: Xu et al. (2023). PIDNet Official Repository.
#   https://github.com/XuJiacong/PIDNet
# - SydneyScapes Dataset:
#   https://ses.library.usyd.edu.au/handle/2123/33051

In [2]:
!pip install torch torchvision --quiet
!pip install numpy pillow tqdm matplotlib --quiet

In [3]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if 'sydney' in file.lower() or 'zip' in file.lower():
            print(os.path.join(root, file))

In [4]:
from google.colab import drive
drive.mount('/content/drive')

# import shutil
import zipfile
import os

# shutil.rmtree('/content/sydneyscapes')
# print('Old extraction deleted.')

ZIP_PATH = '/content/drive/MyDrive/AAIRT1/sydneyscapes.zip'
EXTRACT_PATH = '/content/sydneyscapes'

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)
print('Done!')

DATASET_PATH = '/content/sydneyscapes/sydneyscapes'

print('val.txt exists:', os.path.exists(os.path.join(DATASET_PATH, 'val.txt')))
print('Images exist:', os.path.exists(os.path.join(DATASET_PATH, 'leftImg8bit/val/NSWday')))
print('Labels exist:', os.path.exists(os.path.join(DATASET_PATH, 'gtFine/val/NSWday')))

Mounted at /content/drive
Done!
val.txt exists: True
Images exist: True
Labels exist: True


In [6]:
import os

for root, dirs, files in os.walk(EXTRACT_PATH):
    level = root.replace(EXTRACT_PATH, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 4:
        subindent = ' ' * 2 * (level + 1)
        for f in files[:2]:
            print(f'{subindent}{f}')

test_img = os.path.join(EXTRACT_PATH, 'leftImg8bit/val/NSWday/NSWday_000009_345013_leftImg8bit.png')
test_lbl = os.path.join(EXTRACT_PATH, 'gtFine/val/NSWday/NSWday_000009_345013_gtFine_labelIds.png')

print('Image exists:', os.path.exists(test_img))
print('Label exists:', os.path.exists(test_lbl))

sydneyscapes/
  __MACOSX/
    ._sydneyscapes
    sydneyscapes/
      ._gtFine
      ._train.txt
      leftImg8bit/
        ._val
        ._train
        val/
          NSWnight/
          NSWpeople/
          NSWday/
        train/
          NSWnight/
          NSWpeople/
          NSWday/
      gtFine/
        ._val
        ._train
        val/
          NSWnight/
          NSWpeople/
          NSWday/
        train/
          NSWnight/
          NSWpeople/
          NSWday/
  sydneyscapes/
    .DS_Store
    val.txt
    leftImg8bit/
      .DS_Store
      val/
        .DS_Store
        NSWnight/
        NSWpeople/
        NSWday/
      train/
        .DS_Store
        NSWnight/
        NSWpeople/
        NSWday/
    gtFine/
      .DS_Store
      val/
        .DS_Store
        NSWnight/
        NSWpeople/
        NSWday/
      train/
        .DS_Store
        NSWnight/
        NSWpeople/
        NSWday/
Image exists: False
Label exists: False


In [7]:
import os
import time
import numpy as np
from PIL import Image
import torch
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [8]:
with open(os.path.join(DATASET_PATH, 'val.txt'), 'r') as f:
    val_files = [line.strip() for line in f.readlines()]

print(f'validation images: {len(val_files)}')

Total validation images: 152


In [9]:
CITYSCAPES_TRAIN_IDS = {
    7: 0, 8: 1, 11: 2, 12: 3, 13: 4,
    17: 5, 19: 6, 20: 7, 21: 8, 22: 9,
    23: 10, 24: 11, 25: 12, 26: 13, 27: 14,
    28: 15, 31: 16, 32: 17, 33: 18
}

def label_to_train_id(label_img):
    label_array = np.array(label_img)
    train_id_array = np.full_like(label_array, 255)
    for label_id, train_id in CITYSCAPES_TRAIN_IDS.items():
        train_id_array[label_array == label_id] = train_id
    return train_id_array

def compute_miou(preds, labels, num_classes=19, ignore_index=255):
    iou_list = []
    for cls in range(num_classes):
        pred_mask = preds == cls
        label_mask = labels == cls
        valid_mask = labels != ignore_index
        intersection = (pred_mask & label_mask & valid_mask).sum()
        union = ((pred_mask | label_mask) & valid_mask).sum()
        if union == 0:
            continue
        iou_list.append(intersection / union)
    return np.mean(iou_list) if iou_list else 0.0

Class mapping and mIoU function ready.


In [10]:
transform = transforms.Compose([
    transforms.Resize((512, 1024)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
print('Transform ready.')

Transform ready.


In [11]:
from torchvision.models.segmentation import deeplabv3_resnet101
from torchvision.models.segmentation import DeepLabV3_ResNet101_Weights

print('loading DeepLabv3+ ...')
deeplabv3 = deeplabv3_resnet101(
    weights=DeepLabV3_ResNet101_Weights.COCO_WITH_VOC_LABELS_V1
)
deeplabv3 = deeplabv3.to(device)
deeplabv3.eval()
print('DeepLabv3+ loaded successfully.')

Loading DeepLabv3+...
Downloading: "https://download.pytorch.org/models/deeplabv3_resnet101_coco-586e9e4e.pth" to /root/.cache/torch/hub/checkpoints/deeplabv3_resnet101_coco-586e9e4e.pth


100%|██████████| 233M/233M [00:01<00:00, 180MB/s]


DeepLabv3+ loaded successfully.


In [12]:
deeplabv3_ious = []
deeplabv3_times = []
skipped = 0

print('running DeepLabv3+ on SydneyScapes validation set...')

for file_id in tqdm(val_files):
    subset = file_id.split('/')[0]
    base_name = file_id.split('/')[1]

    img_path = os.path.join(DATASET_PATH, 'leftImg8bit/val', subset, base_name + '_leftImg8bit.png')
    label_path = os.path.join(DATASET_PATH, 'gtFine/val', subset, base_name + '_gtFine_labelIds.png')

    if not os.path.exists(img_path) or not os.path.exists(label_path):
        skipped += 1
        continue

    img = Image.open(img_path).convert('RGB')
    label = Image.open(label_path)
    label_array = label_to_train_id(label)

    input_tensor = transform(img).unsqueeze(0).to(device)

    start = time.time()
    with torch.no_grad():
        output = deeplabv3(input_tensor)['out']
    elapsed = time.time() - start

    pred = output.argmax(1).squeeze().cpu().numpy()
    pred_resized = np.array(
        Image.fromarray(pred.astype(np.uint8)).resize(
            (label_array.shape[1], label_array.shape[0]),
            Image.NEAREST
        )
    )

    iou = compute_miou(pred_resized, label_array)
    deeplabv3_ious.append(iou)
    deeplabv3_times.append(elapsed)

deeplabv3_miou = np.mean(deeplabv3_ious) * 100
deeplabv3_fps = 1.0 / np.mean(deeplabv3_times)

print(f'\nDeepLabv3+ Results:')
print(f'Images evaluated: {len(deeplabv3_ious)} | Skipped: {skipped}')
print(f'mIoU: {deeplabv3_miou:.2f}%')
print(f'FPS: {deeplabv3_fps:.1f}')

Running DeepLabv3+ on SydneyScapes validation set...


100%|██████████| 152/152 [01:22<00:00,  1.84it/s]


DeepLabv3+ Results:
Images evaluated: 152 | Skipped: 0
mIoU: 0.38%
FPS: 45.5
